In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm

import matplotlib.pyplot as plt
%matplotlib inline  

import plotly.graph_objects as go

tfk = tf.keras
tfkl = tfk.layers

In [ ]:
def fractal_function(x, y):
    x = 2*x 
    y = 2*y
    z = np.sin(10 * np.pi * x) * np.cos(10 * np.pi * y) + np.sin(np.pi * (x**2 + y**2))
    z += np.abs(x - y) + (np.sin(5 * x * y) / (0.1 + np.abs(x + y)))
    z *= np.exp(-0.1 * (x**2 + y**2))
    
    # Add noise to z
    noise = np.random.normal(0, 0.1, z.shape)
    z += noise
    
    return z

In [ ]:
X, Y = np.meshgrid(
    np.linspace(-1, 1, 100), 
    np.linspace(-1, 1, 100)
)
Z = fractal_function(X, Y)

In [ ]:
from skopt.space import Space
from skopt.sampler import Halton


n_samples = 150*150
space = Space([(-1.0, 1.0), (-1.0, 1.0)])

sampler = Halton()

x_train = np.array(sampler.generate(space.dimensions, n_samples))
y_train = fractal_function(x_train[:,0], x_train[:,1]).reshape((-1, 1))

In [ ]:
from arnold.layers.core.polynomial.orthogonal import (
    Wilson
)

model = tfk.Sequential([
        tfkl.Reshape((2,)),
        tfkl.Rescaling(scale=1./255., offset=0),
        Wilson(input_dim=2, output_dim=8, degree=3),
        tfkl.LayerNormalization(),
        Wilson(input_dim=8, output_dim=16, degree=3),
        tfkl.LayerNormalization(),
        Wilson(input_dim=16, output_dim=8, degree=3),
        tfkl.LayerNormalization(),
        Wilson(input_dim=8, output_dim=1, degree=2),
    ],
    name="gegenbauer_kan" 
)

model.build((None, 2))
model.compile(
    optimizer=tf.keras.optimizers.Nadam(),
    loss='huber',
    metrics=['mse']
)

model.summary()

EPOCHS = 100
BATCH_SIZE = 256

h = model.fit(
    x_train,
    y_train,
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=1
)

In [ ]:
kernel_size, input_channel, groups, filters = ((2,2), 3, 1, 256) 

# kernel_shape 
t = kernel_size + (input_channel // groups, filters, )

In [ ]:
t, tf.math.reduce_prod(t)

In [ ]:
t[1:]

In [ ]:
(-1,) + t

In [ ]:
from arnold.layers.convolutional.conv_base import ConvBase

In [ ]:
test = ConvBase(
    filters=256,
    kernel_size=(2,2),
    strides=(1, 1),
    padding="valid",
    data_format=None,
    dilation_rate=(1, 1),
    groups=1,
    activation=None,
    use_bias=False,
    kernel_initializer=None,
    bias_initializer=None,
    kernel_regularizer=None,
    bias_regularizer=None,
    activity_regularizer=None,
    kernel_constraint=None,
    bias_constraint=None,
)

In [ ]:
test

In [ ]:
test.build((None, 28,28,1))

In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import tensorflow_datasets as tfds

import matplotlib.pyplot as plt
%matplotlib inline  

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import pprint

tfk = tf.keras
tfkl = tfk.layers

from arnold.layers.convolutional.conv_base import ConvBase
from arnold.layers.core.polynomial.orthogonal import Jacobi, Chebyshev1st, AskeyWilson

In [ ]:
from arnold.layers import Jacobi

In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import tensorflow_datasets as tfds

import matplotlib.pyplot as plt
%matplotlib inline  

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import pprint

tfk = tf.keras
tfkl = tfk.layers

from arnold.layers.convolutional.conv_base import ConvBase
from arnold.layers.core.polynomial.orthogonal import Jacobi, Chebyshev1st, AskeyWilson

BATCH_SIZE = 1024

(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)



kan_conv_model = tfk.Sequential([
        tfkl.Rescaling(scale=1./127.5, offset=-1.),
        tfkl.Reshape((28,28,1)),
        ConvBase(
            filters=32,
            kernel_size=(5,5),
            strides=(3, 3),
            padding="VALID",
            data_format="channels_last",
            dilation_rate=(1, 1),
            groups=1,
            kernel_type="Bump",
            # kernel_args={"degree": 2}
        ),
        tfkl.LayerNormalization(),        
        # ConvBase(
        #     filters=32,
        #     kernel_size=(5,5),
        #     strides=(2, 2),
        #     padding="VALID",
        #     data_format="channels_last",
        #     dilation_rate=(1, 1),
        #     groups=1,
        #     kernel_type="Bump",
        #     # kernel_args={"degree": 3}
        # ),
        # tfkl.LayerNormalization(),
        ConvBase(
            filters=16,
            kernel_size=(5,5),
            strides=(2, 2),
            padding="VALID",
            data_format="channels_last",
            dilation_rate=(1, 1),
            groups=1,
            kernel_type="Chebyshev3rd",
            kernel_args={"degree": 2}
        ),
        tfkl.LayerNormalization(rms_scaling=True),
        tfkl.Flatten(),
        Chebyshev1st(input_dim=64, output_dim=10, degree=3),
        tfkl.Softmax()
    ],
    name="kan_conv_model" 
)

kan_conv_model.build((None, 28, 28, 1))
kan_conv_model.summary()

kan_conv_model.compile(
    optimizer=tf.keras.optimizers.Nadam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


EPOCHS = 20 

kan_conv_model.fit(
        ds_train,
        epochs=EPOCHS, 
        shuffle=True,
        verbose=1
)

In [ ]:
kan_conv_model.evaluate(ds_test)

In [ ]:
conv_model = tfk.Sequential([
        tfkl.Rescaling(scale=1./255., offset=0.),
        tfkl.Reshape((32,32,3)),
        tfkl.Conv2D(
            filters=64,
            kernel_size=(7,7),
            strides=(3, 3),
            padding="VALID",
            data_format="channels_last",
            dilation_rate=(1, 1),
            groups=1,
            activation="relu"
        ),
        tfkl.LayerNormalization(),
        tfkl.Conv2D(
            filters=32,
            kernel_size=(3,3),
            strides=(2, 2),
            padding="VALID",
            data_format="channels_last",
            dilation_rate=(1, 1),
            groups=1,
            activation="relu"
        ),
        tfkl.LayerNormalization(rms_scaling=True),
        tfkl.Flatten(),
        tfkl.Dense(units=10, activation=None),
        tfkl.Softmax()
    ],
    name="conv_model" 
)


conv_model.build((None, 32, 32, 3))
conv_model.summary()

conv_model.compile(
    optimizer=tf.keras.optimizers.Nadam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


EPOCHS = 20 

conv_model.fit(
        ds_train,
        epochs=EPOCHS, 
        shuffle=True,
        verbose=1
)

In [ ]:
conv2d_model = tfk.Sequential([
        tfkl.Rescaling(scale=1./127.5, offset=-1),
        tfkl.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        tfkl.MaxPooling2D(pool_size=(2, 2)),
        tfkl.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        tfkl.MaxPooling2D(pool_size=(2, 2)),
        tfkl.Flatten(),
        tfkl.Dropout(0.5),
        tfkl.Dense(10, activation="softmax"),
    ],
    name="conv2d_model" 
)

conv2d_model.build((None, 28, 28, 1))
conv2d_model.summary()

conv2d_model.compile(
    optimizer=tf.keras.optimizers.Nadam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

conv2d_model.summary()

In [ ]:
BATCH_SIZE = 512

(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)

In [ ]:
EPOCHS = 20

In [ ]:
kan_h = kan_conv_model.fit(
        ds_train,
        epochs=EPOCHS, 
        shuffle=True,
        verbose=1
)

In [ ]:
import pandas as pd
pd.DataFrame(kan_h.history).plot(figsize=(8,5), title='KANconv')
plt.show()

In [ ]:
kan_conv_model.evaluate(ds_test)

In [ ]:
conv_h = conv2d_model.fit(
        ds_train,
        epochs=EPOCHS, 
        shuffle=True,
        verbose=1
)

In [ ]:
import pandas as pd
pd.DataFrame(conv_h.history).plot(figsize=(8,5), title='Conv2D')
plt.show()

In [ ]:
conv2d_model.evaluate(ds_test)

# patch extraction

In [ ]:
imgs = next(ds_train.as_numpy_iterator())[0]

In [ ]:
imgs = tf.random.uniform((1,6,6,3))

In [ ]:
tf.reshape(imgs[:,:,:,0], (1,6,6,1))

In [ ]:
imgs

In [ ]:
tf.expand_dims(imgs[...,0], axis=-1)

In [ ]:
        list(map( 
            # extract all image patches per channel
            # (-1), num_patches_w, num_pachtes_H, kernel_size[0] * kernel_size[1] * channels)
            lambda c: tf.image.extract_patches(
                images=tf.expand_dims(imgs[...,c], axis=-1),
                sizes=(1,3,3,1),
                strides=(1,3, 3, 1),
                padding="VALID",
                rates=[1, 1, 1, 1],
            ),
            range(3)
        ))

In [ ]:
imgs = tf.random.uniform((2,6,6,3))

list(
    map(
        # reshape to (-1, kernel_size[0] * kernel_size[1] * channels)
        lambda t: tf.reshape(t, (-1, 3*3*1)),
        map( 
            # extract all image patches per channel
            # return_shape: (-1), num_patches_w, num_pachtes_H, kernel_size[0] * kernel_size[1] * channels)
            lambda c: tf.image.extract_patches(
                images=tf.expand_dims(imgs[...,c], axis=-1),
                sizes=(1,3,3,1),
                strides=(1,3, 3, 1),
                padding="VALID",
                rates=[1, 1, 1, 1],
            ),
            range(3)
        )
    )
)

In [1]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import tensorflow_datasets as tfds

import matplotlib.pyplot as plt
%matplotlib inline  

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import pprint

tfk = tf.keras
tfkl = tfk.layers


BATCH_SIZE = 512

(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)



from arnold.layers.core.rational_functions import Laurent
test = tfk.Sequential([
    tfkl.Reshape(target_shape=(784, )),
    tfkl.Rescaling(scale=1./255., offset=0.),
    Laurent(input_dim=784, output_dim=10, degree=5),
    # tfkl.LayerNormalization(),  
    # GaussianRBF(input_dim=64, output_dim=32),
    # tfkl.LayerNormalization(),  
    # GaussianRBF(input_dim=32, output_dim=10),
    tfkl.Softmax()
])
test.build((None, 28, 28, 1))
test.summary()

test.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS = 20

test.fit(
    ds_train,
    epochs=EPOCHS, 
    shuffle=True,
    verbose=1
) 

test.evaluate(ds_test)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ laurent (Laurent)               │ (None, 10)             │       133,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 133,280 (520.62 KB)

 Trainable params: 133,280 (520.62 KB)

 Non-trainable params: 0 (0.00 B)

...

/Users/resc01-admin/Coding/aspekteins/arnold/tf_osx_2.15.0/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:664: UserWarning: Gradients do not exist for variables ['polynomial_coefficients'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


  4/118 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0942 - loss: nan

I0000 00:00:1723193564.708678 64529629 service.cc:146] XLA service 0x2de14c220 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1723193564.708697 64529629 service.cc:154]   StreamExecutor device (0): Host, Default Version
2024-08-09 10:52:44.713838: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1723193564.767484 64529629 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


118/118 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.1000 - loss: nan
Epoch 2/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.0985 - loss: nan
Epoch 3/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.0978 - loss: nan
Epoch 4/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.0987 - loss: nan
Epoch 5/20
 28/118 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.1025 - loss: nan

KeyboardInterrupt: 

In [1]:
(0.5 - test.layers[2].grid)

NameError: name 'test' is not defined

array([[[1],
        [2],
        [3],
        [4],
        [5]]])

In [127]:
r = tf.random.uniform((1,5,3), dtype=tf.float32)
eps = tf.constant(np.array([0,1,2.,3.,4.]), dtype=tf.float32)
eps = tf.reshape(eps, (1,5,1))

# tf.math.multiply(tf.reshape(r, (1,5,1,3)),eps)
r*eps


<tf.Tensor: shape=(1, 5, 3), dtype=float32, numpy=
array([[[0.        , 0.        , 0.        ],
        [0.5028274 , 0.9320538 , 0.7062664 ],
        [1.081006  , 1.3873069 , 1.2112079 ],
        [2.8704705 , 2.3446155 , 1.9820527 ],
        [0.62053156, 2.732274  , 1.1646171 ]]], dtype=float32)>

In [128]:
r

<tf.Tensor: shape=(1, 5, 3), dtype=float32, numpy=
array([[[0.8273088 , 0.1363424 , 0.9129566 ],
        [0.5028274 , 0.9320538 , 0.7062664 ],
        [0.540503  , 0.69365346, 0.60560393],
        [0.95682347, 0.7815385 , 0.6606842 ],
        [0.15513289, 0.6830685 , 0.29115427]]], dtype=float32)>

In [ ]:
x = tf.random.normal((1,7,)) 
x = tf.reshape(x, (-1, 7, 1))
tf.math.l2_normalize(x - test.layers[2].grid), tf.math.sqrt((x - test.layers[2].grid)**2)

In [ ]:
test.layers[2].scale
x = tf.ones((128,784,))


tf.repeat(tf.expand_dims(test.layers[2].scale, axis=0), (tf.shape(x)[0]), axis=0)



In [ ]:
test = tfk.Sequential([
    tfkl.Reshape((24*24,25,)),
    tfkl.Conv2D(filters=3225, output_dim=32, degree=3)
])
test.build((None, 24, 24, 25))
test.summary()

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=False,
    with_info=True,
)
ds_train.take(1).cache().repeat()
img = (next(iter(ds_train))['image'])

In [ ]:
imgs = tf.random.uniform((7,11,11,1))

tf.image.extract_patches(
    images=imgs,
    sizes=(1,5,5,1),
    strides=(1,4, 4, 1),
    padding="VALID",
    rates=[1, 1, 1, 1],
)

In [ ]:
import tensorflow as tf

tfk = tf.keras
tfkl = tfk.layers

from arnold.layers import *
test = tfk.Sequential([
    tfkl.Reshape((2*2*25,)),
    Shannon(input_dim=2*2*25, output_dim=32),
])
test.build((None, 2, 2, 25))
test.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
test.summary()

Assume a KAN layer with
- input_dim = m
- output_dim = n
- degree = d

Thus we have poly_coeffs.shape = (m,n,d+1)

Instead of storing the entire (m * n* (d+1)) sized poly_coeffs tensor, we could opt for a "decomposed" representation.

E.g. Tucker representation: T == (m, n, (d+1)) = C x_1 U_1 x_2 U_2 x_3 U_3

- C = (r_1, r_2, r_3)
- U_1 = (m, r_1)
- U_2 = (n, r_2)
- U_3 = (d+1, r_3)



In [ ]:
import opt_einsum as oe

In [ ]:
b, i, o, d = (1, 1024*1024*3, 128*128*64, 12)
x, y, z = (20, 10, 11)

basis = tf.convert_to_tensor(tf.random.uniform((b, i, d)))
coeff = tf.convert_to_tensor(tf.random.uniform((i, o, d)))
core = tf.convert_to_tensor(tf.random.uniform((x, y, z)))

A = tf.random.uniform((i, x))
B = tf.random.uniform((o, y))
C = tf.random.uniform((d, z))

basis.shape, coeff.shape, core.shape, A.shape, B.shape, C.shape

print('basis.size ', np.prod(basis.shape))
print('coeff.size ', np.prod(coeff.shape))
print('tucker size ', np.prod(core.shape) + np.prod(A.shape) + np.prod(B.shape) + np.prod(C.shape))

In [ ]:
%%timeit
tf.einsum(
    'bid,iod->bo', 
    basis,
    coeff
)

In [ ]:
%%timeit
tf.einsum(
    # basis * coeff
    'bid,iod->bo', 
    basis, 
    tf.einsum(
        'xyz,dz->xyd', 
        tf.einsum(
            'xyz,oy->xoz', 
            tf.einsum(
                'xyz,ix->iyz', 
                core, A
            ), 
        B), 
    C)
)

In [ ]:
%%timeit
tf.einsum(
    'bid,xyz,ix,oy,dz->bo', 
    basis, core, A, B, C,
    optimize='branch-all'
)

In [ ]:
np_einsum_path = np.einsum_path(
    'bid,xyz,ix,oy,dz->bo', 
    basis, core, A, B, C,
    optimize='optimal'
)

In [ ]:
np_einsum_path

In [ ]:
%%timeit
np.einsum(
    'bid,xyz,ix,oy,dz->bo', 
    basis, core, A, B, C,
    optimize=np_einsum_path[0]
)

In [ ]:
path_info = oe.contract_path(
    'bid,xyz,ix,oy,dz->bo', 
    basis, core, A, B, C,
    optimize='optimal', memory_limit=None
)

print(path_info)

In [ ]:
%%timeit
oe.contract(
    'bid,xyz,ix,oy,dz->bo', 
    basis, core, A, B, C,
    optimize='optimal', memory_limit=None, backend='tensorflow'
)


In [ ]:
def n_mode_product(x, u, n):
    n = int(n)
    # We need one letter per dimension
    # (maybe you could find a workaround for this limitation)
    if n > 26:
        raise ValueError('n is too large.')
    ind = ''.join(chr(ord('a') + i) for i in range(n))
    exp = f'{ind}K...,JK->{ind}J...'
    return tf.einsum(exp, x, u)

# Test
x = tf.ones((2, 3, 4, 5))
u = tf.ones((6, 4))
n = 2  # n is zero-based here
out = n_mode_product(x, u, n)
print(out.shape)
# (2, 3, 6, 5)

In [ ]:
kan_conv_model.layers[6].poly_basis()

# Implict tucker decomposed KANs

In [ ]:
import tensorflow as tf

from arnold.layers.core.kan_base import KANBase

tfk = tf.keras
tfkl = tfk.layers

class TuckerKAN(KANBase):

    def __init__(
            self, 
            *args,
            degree,
            core_ranks,
            **kwargs):
        
        super().__init__(*args, **kwargs)

        self.degree = degree
        self.r1, self.r2, self.r3 = core_ranks

        self.poly_coeffs_core = self.add_weight(
            shape=(self.r1, self.r2, self.r3),
            initializer=tfk.initializers.RandomNormal(
                mean=0.0, 
                stddev=(1.0 / (self.input_dim * (self.degree + 1)))
            ),
            constraint=None,
            regularizer=None,
            trainable=True,
            name='polynomial_coefficients_core'     
        )

        self.poly_coeffs_A = self.add_weight(
            shape=(self.input_dim, self.r1),
            initializer=tfk.initializers.RandomNormal(
                mean=0.0, 
                stddev=(1.0 / (self.input_dim * (self.degree + 1)))
            ),
            constraint=None,
            regularizer=None,
            trainable=True,
            name='polynomial_coefficients_A'     
        )

        self.poly_coeffs_B = self.add_weight(
            shape=(self.output_dim, self.r2),
            initializer=tfk.initializers.RandomNormal(
                mean=0.0, 
                stddev=(1.0 / (self.input_dim * (self.degree + 1)))
            ),
            constraint=None,
            regularizer=None,
            trainable=True,
            name='polynomial_coefficients_B'     
        )

        self.poly_coeffs_C = self.add_weight(
            shape=(self.degree + 1, self.r3),
            initializer=tfk.initializers.RandomNormal(
                mean=0.0, 
                stddev=(1.0 / (self.input_dim * (self.degree + 1)))
            ),
            constraint=None,
            regularizer=None,
            trainable=True,
            name='polynomial_coefficients_C'     
        )

    def call(self, inputs):
        # Normalize x to [-1, 1] using tanh
        x = tf.tanh(inputs) if self.tanh_x else inputs
        x = tf.reshape(x, (-1, self.input_dim))

        # Compute the polynom interpolation with y.shape=(batch_size, output_dim)
        return tf.einsum(
           'bid,xyz,ix,oy,dz->bo', 
            self.poly_basis(x), self.poly_coeffs_core, self.poly_coeffs_A, self.poly_coeffs_B, self.poly_coeffs_C,
            optimize='auto'
        )

    @tf.function
    def poly_basis(self, x):
        """
        Evaluate Legendre basis polynomials for given `x`."""

        # :math:`P_{0}(x) = 1`
        legendre_basis = [ tf.ones_like(x) ]
        
        if self.degree > 0:
            # :math:`P_{1}(x) = x`
            legendre_basis.append(x)

        for n in range(2, self.degree + 1):
            # :math:`P_{n+1}(x) = \frac{(2n + 1) * x * P_{n}(x) - n * P_{n-1}(x)}{n+1}` when n >= 1
            legendre_basis.append((((2 * n - 1) * x * legendre_basis[n-1]) - ((n - 1) * legendre_basis[n-2])) / n)

        return tf.stack(legendre_basis, axis=-1)


In [ ]:
tucker_legendre_kan = tfk.Sequential([
    tfkl.Reshape(target_shape=(784, )),
    tfkl.Rescaling(scale=1./255., offset=0.),
    TuckerKAN(input_dim=784, output_dim=10, degree=12, core_ranks=(10,10,12)),
    tfkl.LayerNormalization(),  
    tfkl.Softmax()
    ],
)

tucker_legendre_kan.build((None, 28, 28, 1))
tucker_legendre_kan.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
tucker_legendre_kan.summary()

In [ ]:
EPOCHS = 20


tucker_legendre_kan.fit(
        ds_train,
        epochs=EPOCHS, 
        shuffle=True,
        verbose=1
    ) 

In [ ]:
BATCH_SIZE = 512
(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)

In [ ]:
import pandas as pd

pd.DataFrame(tucker_legendre_h.history).plot(figsize=(8,5), title='tucker legende kan')
plt.show()

In [ ]:
tucker_legendre_kan.evaluate(ds_test)

In [ ]:
data = tucker_legendre_kan.layers[2].poly_coeffs_B.numpy()
plt.imshow(data)
plt.title("output - core")
heatmap = plt.pcolor(data) 
plt.colorbar(heatmap)

In [ ]:
data = tucker_legendre_kan.layers[2].poly_coeffs_C.numpy()
plt.imshow(data)
plt.title("degree - core")
heatmap = plt.pcolor(data) 
plt.colorbar(heatmap)

In [ ]:
for i in range(3):
    data = tf.reshape(tucker_legendre_kan.layers[2].poly_coeffs_A, (28,28,-1))[..., i]
    plt.figure()
    plt.imshow(data)
    plt.title("pixel - core")
    heatmap = plt.pcolor(data) 
    plt.colorbar(heatmap)
    plt.show()



In [ ]:
from arnold.layers.core.polynomial import Laurent

legendre_kan = tfk.Sequential([
    tfkl.Reshape(target_shape=(784, )),
    tfkl.Rescaling(scale=1./255., offset=0.),
    Laurent(input_dim=784, output_dim=10, degree=3),
    tfkl.LayerNormalization(),  
    tfkl.Softmax()
    ],
)

legendre_kan.build((None, 28, 28, 1))
legendre_kan.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
legendre_kan.summary()

In [ ]:
EPOCHS = 100 


legendre_h = legendre_kan.fit(
        ds_train,
        epochs=EPOCHS, 
        shuffle=True,
        verbose=1
    ) 

In [ ]:
import pandas as pd

pd.DataFrame(legendre_h.history).plot(figsize=(8,5), title='legende kan')
plt.show()

In [ ]:
legendre_kan.evaluate(ds_test)

In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import tensorflow_datasets as tfds

import matplotlib.pyplot as plt
%matplotlib inline  

BATCH_SIZE = 32
(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)

In [ ]:
import tensorflow as tf

tfk = tf.keras
tfkl = tfk.layers


from arnold.layers import *
test = tfk.Sequential([
    tfkl.Reshape(target_shape=(784, )),
    tfkl.Rescaling(scale=1./255., offset=0.),
    Chebyshev4th(input_dim=784, output_dim=10, degree=5, decompose_weights=True, core_ranks=(10,10,12)),
    tfkl.Softmax()
])
test.build((None, 28, 28, 1))
test.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
test.summary()

In [ ]:
%load_ext tensorboard
import datetime

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

EPOCHS = 10
test.fit(
        ds_train,
        epochs=EPOCHS, 
        shuffle=True,
        verbose=1,
         callbacks=[tensorboard_callback]
    ) 

In [ ]:
%tensorboard

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=2, cols=1)

fig.add_trace(
    go.Heatmap(
        z=test.layers[2].poly_coeffs_C.numpy(),
        showlegend=False
        # text=dict(x="Core R3", y="Poly degrees", color="Strength"),
        # text_auto=True,
        # aspect="auto"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Heatmap(
        z=test.layers[2].poly_coeffs_B.numpy(),
        showlegend=False
        # labels=dict(x="Core R2", y="Outputs", color="Strength"),
        # text_auto=True,
        # aspect="auto"
    ),
    row=2, col=1
)

In [ ]:
tf.reduce_mean(test.layers[2].poly_coeffs_C), tf.math.reduce_variance(test.layers[2].poly_coeffs_C)

In [ ]:
foo

In [ ]:
foo = tf.sparse.from_dense(test.layers[2].poly_coeffs_C - tf.reduce_mean(test.layers[2].poly_coeffs_C))

In [ ]:
X.shape, Y.shape, Z.shape

In [ ]:
X.flatten()

In [ ]:
Y.flatten()

In [ ]:
Z.flatten()

In [ ]:
tf.reshape(test.layers[2].poly_coeffs_A, (28, 28, 10)).numpy()[X,Y,Z].flatten()

In [ ]:
X, Y, Z = np.mgrid[0:28, 0:28, 0:10]

fig = go.Figure(go.Volume(
        x=X.flatten(),
        y=Y.flatten(),
        z=Z.flatten(),
        value=tf.reshape(test.layers[2].poly_coeffs_A, (28, 28, 10)).numpy()[X,Y,Z].flatten()
    )
)
fig.show()

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=3, cols=1)

fig.add_trace(
    go.Heatmap(
        z=test.layers[2].poly_coeffs_C.numpy(),
        showlegend=False
        # text=dict(x="Core R3", y="Poly degrees", color="Strength"),
        # text_auto=True,
        # aspect="auto"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Heatmap(
        z=test.layers[2].poly_coeffs_B.numpy(),
        showlegend=False
        # labels=dict(x="Core R2", y="Outputs", color="Strength"),
        # text_auto=True,
        # aspect="auto"
    ),
    row=2, col=1
)

X, Y, Z = np.mgrid[0:28, 0:28, 0:11]
fig.add_trace(
    go.Volume(
        x=X.flatten(),
        y=Y.flatten(),
        z=Z.flatten(),
        value=test.layers[2].poly_coeffs_A.numpy().flatten()
    ),
    row=3, col=1
)

In [ ]:
poch = lambda x, m: tf.math.divide(
    tf.math.exp(tf.math.lgamma(x + m)),
    tf.math.exp(tf.math.lgamma(x))
)


poch_tf = tf.function(
    lambda x, m: tf.math.divide(
        tf.math.exp(tf.math.lgamma(x + m)),
        tf.math.exp(tf.math.lgamma(x))
    )
)

In [ ]:
%%timeit
poch(0.5,7)


In [ ]:
%%timeit
poch_tf(0.5,7)

In [ ]:
a, b = (0.5, -0.012)
degree = 3
AB = tf.range(0, degree +1, delta=1, dtype=tf.float32)
tf.map_fn(lambda n: poch_tf(a+b, n), AB)

In [ ]:
x = tf.random.uniform((5,))
AB = tf.range(0, degree +1, delta=1, dtype=tf.float32)
tf.map_fn(lambda n: poch_tf(a+x, n), AB)
# shape: (degree, x)

In [ ]:
AB

In [ ]:
a+x

In [ ]:
tf.map_fn(lambda n: poch_tf(a+x, n), AB)

In [ ]:
a, b, c, d = (0.5, -0.012, 1.7, 2/3)
degree = 3
AB = tf.range(0, degree +1, delta=1, dtype=tf.float32)
a_b = tf.math.reduce_prod(tf.map_fn(lambda n: poch_tf(a+b, n), AB))
a_c = tf.math.reduce_prod(tf.map_fn(lambda n: poch_tf(a+c, n), AB))
a_d = tf.math.reduce_prod(tf.map_fn(lambda n: poch_tf(a+d, n), AB))

In [ ]:
%%timeit
tf.math.reduce_prod([
    tf.math.reduce_prod(tf.map_fn(lambda n: poch_tf(a+b, n), AB)),
    tf.math.reduce_prod(tf.map_fn(lambda n: poch_tf(a+c, n), AB)),
    tf.math.reduce_prod(tf.map_fn(lambda n: poch_tf(a+d, n), AB))
])

In [ ]:
tf.math.divide(
        tf.math.exp(tf.math.lgamma(x + m)),
        tf.math.exp(tf.math.lgamma(x))
    )

In [ ]:
tf.math.reduce_prod([
    tf.math.reduce_prod(tf.map_fn(lambda n: tf.math.divide(tf.math.exp(tf.math.lgamma(a+b + n)), tf.math.exp(tf.math.lgamma(a+b))), AB)),
    tf.math.reduce_prod(tf.map_fn(lambda n: tf.math.divide(tf.math.exp(tf.math.lgamma(a+c + n)), tf.math.exp(tf.math.lgamma(a+c))), AB)),
    tf.math.reduce_prod(tf.map_fn(lambda n: tf.math.divide(tf.math.exp(tf.math.lgamma(a+d + n)), tf.math.exp(tf.math.lgamma(a+d))), AB)),
])

In [ ]:
%%timeit
poch_a_b_n = tf.map_fn(lambda n: tf.math.exp(tf.math.lgamma(a+b + n) - tf.math.lgamma(a+b)), AB)
poch_a_c_n = tf.map_fn(lambda n: tf.math.exp(tf.math.lgamma(a+c + n) - tf.math.lgamma(a+c)), AB)
poch_a_d_n = tf.map_fn(lambda n: tf.math.exp(tf.math.lgamma(a+d + n) - tf.math.lgamma(a+d)), AB)

n,k = (4,3)
tf.math.reduce_prod([
    tf.math.reduce_prod(poch_a_b_n[:k]),
    tf.math.reduce_prod(poch_a_c_n[:k]),
    tf.math.reduce_prod(poch_a_d_n[:k]),
])

In [ ]:
a, b, c, d = (0.5, -0.012, 1.7, 2/3)
ab = a+b
ac = a+c
ad = a+d

In [ ]:
%%timeit
a+b+c+d

In [ ]:
%%timeit
ab+c+d

In [ ]:
tf.range(0,5, dtype=tf.float32) * tf.random.uniform((5,))

In [ ]:
a, b, c, d = (0.5, -0.012, 1.7, 2/3)
degree = 3

term1 = (a + b)
term2 = (a + c)
term3 = (a + d)

term = tf.stack([term1,term2,term3])
tf.map_fn(
    lambda k: tf.math.exp(tf.math.lgamma(term + k) - tf.math.lgamma(term)),
    tf.range(0, degree + 1, delta=1, dtype=tf.float32)
)

In [ ]:
a, b, c, d = (0.5, -0.012, 1.7, 2/3)
n = 5 + 1

var_terms = tf.stack([
    a + b,
    a + c,
    a + d,
    a + b + c + d + degree,
])
poch_var_terms = tf.map_fn(
    lambda k: tf.math.exp(tf.math.lgamma(var_terms + k) - tf.math.lgamma(var_terms)),
    tf.range(1, n, delta=1, dtype=tf.float32)
)

poch_var_terms

In [ ]:
poch_var_terms[0] * poch_var_terms[1] * poch_var_terms[2]

In [ ]:
tf.math.reduce_prod(poch_var_terms[-1,0:3])

In [ ]:
degree = 4 + 1
tf.map_fn(
    lambda k: tf.math.exp(tf.math.lgamma((degree + 1 - k) + k) - tf.math.lgamma(degree + 1 - k)),
    tf.range(0, degree + 1, delta=1, dtype=tf.float32)
)

In [ ]:
tf.math.log(0.5)

In [ ]:
tf.math.lgamma(0.)

In [ ]:
x = -0.9*tf.ones((7,))
a = 10.5
degree = 4

x_terms = tf.stack([
            a - x,
            a + x,
])
poch_x_terms = tf.map_fn(
    lambda k: tf.math.exp(tf.math.lgamma(x_terms + k) - tf.math.lgamma(x_terms)),
    tf.range(1, degree + 1, delta=1, dtype=tf.float32)
)

var_terms = tf.stack([
    a + b,
    a + c,
    a + d,
    a + b + c + d + degree,
])
poch_var_terms = tf.map_fn(
    lambda k: tf.math.exp(tf.math.lgamma(var_terms + k) - tf.math.lgamma(var_terms)),
    tf.range(1, n, delta=1, dtype=tf.float32)
)

poch_x_terms

In [ ]:
(poch_x_terms[:,0,:]*poch_x_terms[:,1,:]) * var_terms[0]

In [ ]:
x_terms = tf.stack([
            -(degree + 1.),
])
tf.map_fn(
    lambda k: tf.math.exp(tf.math.lgamma(x_terms + k) - tf.math.lgamma(x_terms)),
    tf.range(1, degree + 1, delta=1, dtype=tf.float32)
)

In [ ]:
tf.random.uniform((7,))

In [ ]:
tf.random.uniform((7,))

In [ ]:
tf.random.uniform((7,))

In [ ]:
tf.random.uniform((7,))

In [ ]:
tf.random.uniform((7,))

In [ ]:
tf.random.normal((7,))

In [ ]:
n = 4 + 1
tf.math.multiply(
    tf.map_fn(
        lambda k: n-k+1,
        tf.range(1, n, delta=1, dtype=tf.float32)
    ),
    tf.map_fn(
        lambda k: (-1)**k,
        tf.range(1, n, delta=1, dtype=tf.float32)
    )
)

In [ ]:
tf.map_fn(
    lambda k: (-1)**k,
    tf.range(1, n, delta=1, dtype=tf.float32)
)

In [ ]:
x = tf.random.normal((7, 13))
a = 0.1
b = 2.5
c = 1/3
d = 0.25
n = 10 + 1

# get poch(a+b; k) for all 0 < k <= degree+1
# get poch(a+c; k) for all 0 < k <= degree+1
# get poch(a+d; k) for all 0 < k <= degree+1
# get poch(a+b+c+d+n-1; k) for all 0 < k <= degree+1
var_terms = tf.stack([
    a + b,
    a + c,
    a + d,
    a + b + c + d + n - 1
])

poch_var_terms = tf.map_fn(
    lambda k: tf.math.exp(tf.math.lgamma(var_terms + k) - tf.math.lgamma(var_terms)),
    tf.range(1, n, delta=1, dtype=tf.float32)
)

# get !(k) for all 0< k <= n
k_factorials = tf.map_fn(
    lambda k: tf.exp(tf.math.lgamma(k + 1)),
    tf.range(1, n, delta=1, dtype=tf.float32)
)

# get poch(-n; k) for all 0 < k <= n
poch_n = tf.math.multiply(
    tf.map_fn(
        lambda k: n-k+1,
        tf.range(1, n, delta=1, dtype=tf.float32)
    ),
    tf.map_fn(
        lambda k: (-1)**k,
        tf.range(1, n, delta=1, dtype=tf.float32)
    )
)

# get poch(a - x; k) for all 0 < k <= n
# get poch(a + x; k) for all 0 < k <= n
x_terms = tf.stack([
    a - x,
    a + x
])

poch_x_terms = tf.map_fn(
    lambda k: tf.math.exp(tf.math.lgamma(x_terms + k) - tf.math.lgamma(x_terms)),
    tf.range(1, n, delta=1, dtype=tf.float32)
)

poch_var_terms.shape, poch_n.shape, k_factorials.shape, tf.transpose(poch_x_terms).shape


# * [ (-n)**k * (a+b+c+d+n-1)_{k} ] / [ (a+b)_{k} * (a+c)_{k} * (a+d)_{k} *!(k) ] * (a-x)_{n} * (a+x)_{n}
# term = poch_n               # (-n)**k
# term *= var_terms[3]        # (a+b+c+d+n-1)_{k}
# term /= var_terms[0]        # (a+b)_{n}
# term /= var_terms[1]        # (a+c)_{n}
# term /= var_terms[2]        # (a+d)_{n}
# term /= k_factorials        # !(k)

# term *= poch_x_terms[:,0,:] # (a-x)_{n}
# term *= poch_x_terms[:,1,:] # (ax)_{n}

In [ ]:
# compute hypergeometric series until k=n

term = poch_n               # (-n)**k
term *= var_terms[3]        # (a+b+c+d+n-1)_{k}
term /= var_terms[0]        # (a+b)_{n}
term /= var_terms[1]        # (a+c)_{n}
term /= var_terms[2]        # (a+d)_{n}
term /= k_factorials        # !(k)
term.shape

In [ ]:
poch_x_terms[:,0,:] .shape

In [ ]:
(term * tf.transpose(poch_x_terms[:,0,:]) * tf.transpose(poch_x_terms[:,1,:])).shape

In [ ]:
tf.math.reduce_all(tf.math.greater(tf.random.normal((13,4,7)), 0.0))

In [ ]:
def pochhammer(k: int, x: tf.Tensor):
    """
    pochhammer(k, x) := x * (x + 1) * ... * (x + k -1)
    """
    if tf.math.reduce_all(tf.math.greater(x, 0.0)):
        # all values in x greater than 0, so it is safe to use lgamma
        return tf.math.exp(tf.math.lgamma(x + k)- tf.math.lgamma(x))

pochhammer(4, tf.ones((13,5)))

In [ ]:
@tf.function
def pochhammer(k: int, x: tf.Tensor):
    """
    The Pochhammer symbol (rising factorial) is defined as
    
    pochhammer(k, x) := x * (x + 1) * ... * (x + k -1) = np.prod([(x+i) for i in range(0,k)])

    See also: https://dlmf.nist.gov/5.2#iii

    :param k: The factorial power
    :type k: non-negative integer
    :param x: value to apply pochhammer to
    :type x: tf.Tensor
    :returns: pochhammer(k, x)
    :rtype: tf.Tensor
    """
    assert k >= 0, "Parameter k must be non-negative!"
    if k == 0:
        return tf.ones_like(x)

    if tf.math.reduce_all(tf.math.greater(x, 0.0)):
        # all values in x greater than 0, so it is safe to use lgamma
        return tf.math.exp(tf.math.lgamma(x + k - 1)- tf.math.lgamma(x))
    else:
        # otherwise we need to iterate
        return tf.math.reduce_prod(
            tf.map_fn(
                lambda m: x+m,
                tf.range(0, k, delta=1, dtype=x.dtype)
            ),
            axis=0
        )


In [ ]:
import tensorflow as tf
from typing import List

x = tf.random.normal((5,7))
a = 0.1 * tf.ones_like(x)
b = 2.5 
c = 1/3
d = 0.25
n = 10 

a_s = [-n * tf.ones_like(x), (a+b+c+d+n-1)* tf.ones_like(x), a* tf.ones_like(x) - x, a* tf.ones_like(x) + x]
b_s = [(a+b) * tf.ones_like(x), (a+c) * tf.ones_like(x), (a+d) * tf.ones_like(x)]
z = 1.

In [ ]:
@tf.function
def pochhammer(k: int, x: tf.Tensor):
    """
    The Pochhammer symbol (rising factorial) is defined as
    
    pochhammer(k, x) := x * (x + 1) * ... * (x + k -1) = np.prod([(x+i) for i in range(0,k)])

    See also: https://dlmf.nist.gov/5.2#iii

    :param k: The factorial power
    :type k: non-negative integer

    :param x: value to apply pochhammer to
    :type x: tf.Tensor

    :returns: pochhammer(k, x)
    :rtype: tf.Tensor
    """
    assert k >= 0, "Parameter k must be non-negative!"
    if k == 0:
        return tf.ones_like(x)

    if tf.math.reduce_all(tf.math.greater(x, 0.0)):
        # all values in x greater than 0, so it is safe to use lgamma
        return tf.math.exp(tf.math.lgamma(x + k - 1)- tf.math.lgamma(x))
    else:
        # otherwise we need to iterate
        return tf.math.reduce_prod(
            tf.map_fn(
                lambda m: x+m,
                tf.range(0, k, delta=1, dtype=x.dtype)
            ),
            axis=0
        )

@tf.function
def generalized_hypergeometric(a_s: List[tf.Tensor], b_s: List[tf.Tensor], z: tf.Tensor, num_terms:int):
    """
    Generalized hypergeometric function.
    
    # :math:`{}_{p}F_{q}(a_{1},\ldots ,a_{p};b_{1},\ldots ,b_{q};z)=\sum _{n=0}^{\infty }{\frac {(a_{1})_{n}\cdots (a_{p})_{n}}{(b_{1})_{n}\cdots (b_{q})_{n}}}\,{\frac {z^{n}}{n!}}`

    :param a_s: List of tf.Tensors with same shape
    :type a_s: List[tf.Tensor]

    :param b_s: List of tf.Tensors with same shape
    :type b_s: List[tf.Tensor]

    :param z_s: order-1 tensor
    :type z: tf.Tensor

    :params num_terms: number of summation terms to compute
    :type num_terms: non-negative integer

    :returns: :math:`{}_{p}F_{q}(a_{1},\ldots ,a_{p};b_{1},\ldots ,b_{q};z)`
    :rtype: tf.Tensor
    """

    return tf.math.reduce_sum(
        tf.stack(
            list(map(lambda n:
                tf.math.multiply(
                    tf.math.divide(
                        tf.math.reduce_prod(list(map(lambda t: pochhammer(n, t), a_s)), axis=0),
                        tf.math.reduce_prod(list(map(lambda t: pochhammer(n, t), b_s)), axis=0)
                    ),
                    tf.math.pow(z, n) / tf.math.exp(tf.math.lgamma(n + 1.)
                )),
                range(num_terms)
            ))
        ),
        axis=0
    )


In [ ]:
generalized_hypergeometric(a_s, b_s, z, 1)

In [ ]:
x = tf.random.normal((5,7))
a = 0.1 * tf.ones_like(x)
b = 2.5 
c = 1/3
d = 0.25

tf.stack(list(map(
    lambda degree: generalized_hypergeometric(
        [-degree * tf.ones_like(x), (a+b+c+d+degree-1)* tf.ones_like(x), a* tf.ones_like(x) - x, a* tf.ones_like(x) + x], 
        [(a+b) * tf.ones_like(x), (a+c) * tf.ones_like(x), (a+d) * tf.ones_like(x)], 
        1.0, 
        degree + 1),
    range(3)
)),
axis=-1)

In [ ]:
v = tf.random.normal((13,))
l = 10

In [ ]:
%%timeit
tf.stack([v]*l, axis=0)

In [ ]:
%%timeit
tf.tile(tf.expand_dims(v, axis=0),[l, 1])

# Refactor wavelet

In [ ]:
batch_size, output_dim, input_dim = (13, 8, 16)


x = tf.random.normal((batch_size, input_dim))
translation = tf.random.normal((output_dim, input_dim))
scale = tf.random.normal((output_dim, input_dim))

tf.math.divide(
    tf.math.subtract(
        # (batch_size, 1, input_dim),
        tf.expand_dims(x, axis=1),
        # (batch_size, output_dim, input_dim)
        tf.tile(
            tf.expand_dims(translation, axis=0), 
            [tf.shape(x)[0], 1, 1]
        )   
    ),
    # (batch_size, output_dim, input_dim)
    tf.tile(
        tf.expand_dims(scale, axis=0), 
        [tf.shape(x)[0], 1, 1]
    )             
)

(batch_size, input_dim) - (output_dim, input_dim)

->

(batch_size, input_dim, output_dim)

In [ ]:
batch_size, output_dim, input_dim = (13, 8, 16)


x = tf.random.normal((batch_size, input_dim))
translation = tf.random.normal((output_dim, input_dim))
scale = tf.random.normal((output_dim, input_dim))
W = tf.random.normal((output_dim, input_dim))

A = tf.math.divide(
    tf.math.subtract(
        # (batch_size, 1, input_dim),
        tf.expand_dims(x, axis=1),
        # (batch_size, output_dim, input_dim)
        tf.tile(
            tf.expand_dims(translation, axis=0), 
            [tf.shape(x)[0], 1, 1]
        )   
    ),
    # (batch_size, output_dim, input_dim)
    tf.tile(
        tf.expand_dims(scale, axis=0), 
        [tf.shape(x)[0], 1, 1]
    )             
)
B = tf.math.divide(
    tf.math.subtract(
        # (batch_size, 1, input_dim),
        tf.expand_dims(x, axis=1),
        # (batch_size, output_dim, input_dim)
        tf.expand_dims(translation, axis=0), 
    ),
    # (batch_size, output_dim, input_dim)   
    tf.expand_dims(scale, axis=0), 
)
tf.reduce_sum(B*W, axis=-1).shape


In [ ]:

tf.math.subtract(
        # (batch_size, 1, input_dim),
        tf.expand_dims(x, axis=1),
        # (batch_size, output_dim, input_dim)
            tf.expand_dims(translation, axis=0), 
        )   


In [ ]:
tf.shape(x)

In [ ]:
[tf.expand_dims(translation, axis=0),]

In [ ]:
tf.stack([translation,]*13, axis=0)

In [ ]:
# %%timeit
tf.stack([translation,]*tf.shape(x)[0].numpy(), axis=0)

In [ ]:
%%timeit
tf.tile(
    tf.expand_dims(translation, axis=0), 
    [tf.shape(x)[0], 1, 1]
)  
        

In [ ]:
%%timeit
tf.repeat(
    tf.expand_dims(translation, axis=0), 
    (tf.shape(x)[0]), 
    axis=0
)

In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import tensorflow_datasets as tfds

import matplotlib.pyplot as plt
%matplotlib inline  

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import pprint

tfk = tf.keras
tfkl = tfk.layers


BATCH_SIZE = 32

(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)



from arnold.layers.core.wavelet import Bump, Ricker, Shannon
test = tfk.Sequential([
    tfkl.Reshape(target_shape=(784, )),
    tfkl.Rescaling(scale=1./255., offset=0.),
    Bump(input_dim=784, output_dim=32),
    tfkl.LayerNormalization(),  
    Bump(input_dim=32, output_dim=16),
    tfkl.LayerNormalization(),  
    Bump(input_dim=16, output_dim=10),
    tfkl.Softmax()
])
test.build((None, 28, 28, 1))
test.summary()

test.compile(
    optimizer=tf.keras.optimizers.Nadam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS = 20

test.fit(
    ds_train,
    epochs=EPOCHS, 
    shuffle=True,
    verbose=1
) 

test.evaluate(ds_test)

In [ ]:
t = test.layers[2].translation
s = test.layers[2].scale
x = tf.random.uniform((13,784))

psi = tf.math.divide(
    tf.math.subtract(
        # (batch_size, 1, self.input_dim),
        tf.expand_dims(x, axis=1),
        # (1, output_dim, input_dim)
        t
    ),
    # (1, output_dim, input_dim)
    tf.math.exp(s)
)

coeffs = test.layers[2].wavelet_weights

In [ ]:
%%timeit -r 20
tf.reduce_sum(psi*coeffs, axis=-1)

In [ ]:
%%timeit -r 20
tf.einsum('boi,oi->bo', psi, coeffs, optimize='optimal')

In [ ]:
%%timeit -r 20
tf.einsum('boi,oi->bo', psi, coeffs, optimize='auto')

In [ ]:
%%timeit -r 20
tf.einsum('boi,oi->bo', psi, coeffs, optimize='greedy')

In [ ]:
batch_size, input_dim, output_dim, degree = (512, 784, 128, 12)
poly_basis = tf.random.normal((batch_size, input_dim, degree))
poly_coeffs = tf.random.normal((input_dim, output_dim, degree))


In [ ]:
coeffs_io = tf.transpose(coeffs)

In [ ]:
%%timeit -r 20
tf.einsum('boi,io->bo', psi, coeffs_io, optimize='optimal')

In [ ]:
%%timeit
tf.einsum(
    'bid,iod->bo', 
    poly_basis, 
    poly_coeffs,
    optimize='auto'
)

In [ ]:
batch_size, input_dim, output_dim, degree = (512, 784, 128, 12)
poly_basis = tf.random.normal((batch_size, input_dim, degree))
poly_coeffs = tf.random.normal((input_dim, degree, output_dim))

In [ ]:
%%timeit
tf.einsum(
    'bid,ido->bo', 
    poly_basis, 
    poly_coeffs,
    optimize='auto'
)

In [ ]:
batch_size, input_dim, output_dim, degree = (512, 784, 128, 12)

poly_basis = tf.random.normal((batch_size, input_dim, degree))

poly_coeffs_iod = tf.random.normal((input_dim, output_dim, degree))
poly_coeffs_ido = tf.einsum('iod->ido',poly_coeffs)

poly_coeffs_doi = tf.einsum('iod->doi',poly_coeffs)
poly_coeffs_dio = tf.einsum('iod->dio',poly_coeffs)

poly_coeffs_odi = tf.einsum('iod->odi',poly_coeffs)
poly_coeffs_oid = tf.einsum('iod->oid',poly_coeffs)

In [ ]:
%%timeit -r 20
tf.einsum(
    'bid,ido->bo', 
    poly_basis, 
    poly_coeffs_ido,
    optimize='auto'
)

In [ ]:
%%timeit -r 20
tf.einsum(
    'bid,oid->bo', 
    poly_basis, 
    poly_coeffs_oid,
    optimize='auto'
)

In [ ]:
%%timeit -r 20
tf.einsum(
    'bid,iod->bo', 
    poly_basis, 
    poly_coeffs_iod,
    optimize='auto'
)

In [ ]:
%%timeit -r 10
tf.einsum(
    'bid,doi->bo', 
    poly_basis, 
    poly_coeffs_doi,
    optimize='auto'
)

In [ ]:
%%timeit -r 10
tf.einsum(
    'bid,dio->bo', 
    poly_basis, 
    poly_coeffs_dio,
    optimize='auto'
)

In [ ]:
%%timeit -r 10
tf.einsum(
    'bid,odi->bo', 
    poly_basis, 
    poly_coeffs_odi,
    optimize='auto'
)

In [ ]:
 all((None,1, None))

In [ ]:
import numpy as np


'''
@params:
    A: m × n matrix
    B: n × p matrix
    c: a positive integer
    ps: a list of probabilities (size n)
@return:
    Matrix C and R such that CR ≈ AB
    C: m × c matrix
    R: c × p matrix
'''
def matrix_multi_approx(A: np.array, B: np.array, c: int):
    assert A.shape[1] == B.shape[0], "Shape not match"
    n = A.shape[1]
    probs = compute_ps(A, B)
    C = np.zeros((A.shape[0], c))
    R = np.zeros((c, B.shape[1]))
    for i in range(c):
        it = np.random.choice(n, p=probs)
        C[:, i] = A[:, it]/np.sqrt(c*probs[it])
        R[i, :] = B[it, :]/np.sqrt(c*probs[it])
    return C, R


'''
@params:
    A: m × n matrix
    B: n × p matrix
@return:
    a list of probabilities that minimize E(||AB-CR||^2) (Frobenius norm)
'''
def compute_ps(A: np.array, B: np.array):
    assert A.shape[1] == B.shape[0], "Shape not match"
    n = A.shape[1]
    C =  np.sum(np.array(
        [np.linalg.norm(A[:, k],2) * np.linalg.norm(B[k, :],2) for k in range(n)]
    ))
    return \
        np.array(
            [np.linalg.norm(A[:, k],2) * np.linalg.norm(B[k, :], 2) / C for k in range(n)]
        )


# Construct large scale matrix
x = np.arange(-0.7, 0.701, 0.001)
y = np.arange(-0.7, 0.701, 0.001)
xx, yy = np.meshgrid(x, y, sparse=False)
# calculate z for each x_i and y_j, and store in matrix A
A = B = np.sqrt(1 - xx**2 - yy**3)

# Do approximation
C, R = matrix_multi_approx(A, B, 1401)

# Check the Frobenius norm of AB-CR
AB = A@B
CR = C@R
fro_norm = np.linalg.norm(AB-CR, 'fro')

In [ ]:
fro_norm

In [ ]:
B.shape

In [ ]:
%%timeit
A@B

In [ ]:
%%timeit
C@R

In [ ]:
A = tf.random.uniform((3,5,7))
tf.gather_nd(batch_dims=1,)

# Discrete wavelet

In [ ]:
from abc import abstractmethod
import tensorflow as tf

from arnold.layers.core.kan_base import KANBase

tfk = tf.keras
tfkl = tfk.layers

class DiscreteBump(KANBase):
    """
    Abstract base class for Kolmogorov-Arnold Network layer using wavelets.

    :ivar scale: The (learnable) non-zero, positive scale/dilation parameter for the wavelet transform. Internally stored as log(scale).
    :vartype scale: tf.Tensor

    :ivar translation: The (learnable) translation parameter fro the wavelet transform.
    :vartype translation: tf.Tensor
    
    :ivar wavelet_weights: The learnable wavelet coefficients.
    :vartype wavelet_weights: tf.Tensor
    """

    def __init__(self, 
                 *args,
                 a:float=2.0, b:float=1.0,
                 j:int=2,
                 **kwargs):
        """
        :param input_dim: This layers input size
        :type input_dim: int

        :param output_dim: This layers output size
        :type output_dim: int

        :param tanh_x: Flag indicating whether to normalize any input to [-1, 1] using tanh before further processing.
        :type tanh_x: bool

        :param scale_init: Initial non-zero, positive value for the wavelet scale parameter; defaults to None (log(scale) initialized to HeNormal).
        :type scale_init: non-zero, positive float | None = None

        :param scale_trainable: Flag indicating whether scale is a trainable parameter. Defaults to True
        :type scale_trainable: bool

        :param translation_init: Initial translation value for the wavelet scale parameter; defaults to None (initialized to HeNormal).
        :type translation_init: float | None = None

        :param translation_trainable: Flag indicating whether translation is a trainable parameter. Defaults to True
        :type translation_trainable: bool
        """
        super().__init__(*args, **kwargs)
       
        self.a = a 
        self.b = b
        self.j = j

        self.scale = 2**(-j)

        # shape (1, 2**j -1, inpit_dim)
        self.translation = tf.stack( [tf.expand_dims(tf.map_fn(lambda k: self.scale*k, tf.range(0, 2**self.j, dtype=tf.float32)), axis=0),]*self.input_dim, axis=-1)
        
        # Linear weights for combining outputs
        self.wavelet_weights = self.add_weight(
            shape=(self.output_dim, 2**self.j),
            initializer=tfk.initializers.HeUniform(),
            name='wavelet_weights',
            trainable=True
        )


    @tf.function(autograph=True, jit_compile=True, reduce_retracing=True, experimental_autograph_options=tf.autograph.experimental.Feature.ALL)    
    def mother(self, x):
        eps = 1e-07
        x = tf.clip_by_value(x, -1.0+eps, 1.0-eps)
        return tf.exp(-1.0 / (1 - x**2))
    
    @tf.function(autograph=True, jit_compile=True, reduce_retracing=True, experimental_autograph_options=tf.autograph.experimental.Feature.ALL)
    def call(self, inputs):
        # Normalize x to [-1, 1] using tanh
        x = tf.tanh(inputs) if self.tanh_x else inputs

        x = tf.math.divide(
            tf.math.subtract(
                # (batch_size, 1, self.input_dim),
                tf.expand_dims(x, axis=1),
                # (1, output_dim, input_dim)
                self.translation
            ),
            # (1, output_dim, input_dim)
            self.scale
        )

        daughter_wavelets = self.mother(x)
    
        return tf.einsum(
            'bli,ol->bo', 
            daughter_wavelets, 
            self.wavelet_weights,
            optimize='auto'
        )

In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import tensorflow_datasets as tfds

import matplotlib.pyplot as plt
%matplotlib inline  

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import pprint

tfk = tf.keras
tfkl = tfk.layers


BATCH_SIZE = 512

(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)



test = tfk.Sequential([
    tfkl.Reshape(target_shape=(784, )),
    tfkl.Rescaling(scale=1./255., offset=0.),
    DiscreteBump(input_dim=784, output_dim=32, j=5),
    tfkl.LayerNormalization(),  
    DiscreteBump(input_dim=32, output_dim=16, j=3),
    tfkl.LayerNormalization(),  
    DiscreteBump(input_dim=16, output_dim=10, j=2),
    tfkl.Softmax()
])
test.build((None, 28, 28, 1))
test.summary()

test.compile(
    optimizer=tf.keras.optimizers.Nadam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS = 20

test.fit(
    ds_train,
    epochs=EPOCHS, 
    shuffle=True,
    verbose=1
) 

test.evaluate(ds_test)

In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import tensorflow_datasets as tfds

import matplotlib.pyplot as plt
%matplotlib inline  

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import pprint

tfk = tf.keras
tfkl = tfk.layers


BATCH_SIZE = 512

(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)


from arnold.layers import Bump
test = tfk.Sequential([
    tfkl.Reshape(target_shape=(784, )),
    tfkl.Rescaling(scale=1./255., offset=0.),
    Bump(input_dim=784, output_dim=32),
    tfkl.LayerNormalization(),  
    Bump(input_dim=32, output_dim=16),
    tfkl.LayerNormalization(),  
    Bump(input_dim=16, output_dim=10),
    tfkl.Softmax()
])
test.build((None, 28, 28, 1))
test.summary()

test.compile(
    optimizer=tf.keras.optimizers.Nadam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS = 20

test.fit(
    ds_train,
    epochs=EPOCHS, 
    shuffle=True,
    verbose=1
) 

test.evaluate(ds_test)

In [ ]:
%%timeit
tf.einsum(
    'coi,boi->boi',
    tf.random.normal((1, 13,7)),
    tf.random.normal((11,13,7))
)

In [ ]:
%%timeit
tf.random.normal((1, 13,7)) * tf.random.normal((11,13,7))

In [ ]:
import glob 
from PIL import Image
import os.path

image_sizes = np.array([Image.open(filename).size for filename in glob.glob('/Users/resc01-admin/Downloads/ISIC_2024_Permissive_Training_Input/*.jpg', recursive=False)])

In [ ]:
image_sizes.min(axis=0), image_sizes.max(axis=0), 

In [ ]:
np.mean(image_sizes, axis=0), np.var(image_sizes, axis=0)